# Chaînes — Carnet 07 (C#) : La fonction de Wigner et le signe à N corps

Jumeau C# de [Chaînes-07 — la fonction de Wigner et le signe à N corps (Python)](Chaines-07-Fonction-Wigner-Signe-N-Corps-Python.ipynb) : même fil pédagogique, mêmes entrées, même arithmétique exacte — rejouée ici en `BigInteger` et en rationnel exact, BCL seule.

Série **Chaînes** (distillation du compendium, série #25013). **Chapitre 46 — Théorème 46.1 : « The Wigner function and the many-body sign »** — un **théorème conditionnel**.

> **Standing : CONDITIONNEL.** Le théorème lit ensemble la fonction d'espace des phases et le signe à N corps. Il est conditionnel sur la fonction de Wigner comme cellule de lecture swap — un résultat ouvert qui tient à **quatre des cinq** constats : à la cinquième, le recouvrement de deux distributions *part* du recouvrement d'enregistrements, et l'entrée Wigner devient une glose.

Patron de la série : chaque affirmation est **Mesurée** (rejouée ici en arithmétique exacte — `BigInteger` pour les comptes, `Rationnel` exact pour les coefficients de l'anneau), **Cité** (constante du compendium, non dérivée) ou **Ouverte** (énoncée comme ouverte par la source).


## Contexte et prémices

Le théorème repose sur les enregistrements, la mesure du chapitre 42 (**Théorème 42.1**, carnet Chaînes-03) et l'échange du chapitre 43 (**Théorème 43.1**, carnet Chaînes-04) : la négativité des marginales de Wigner au-delà de deux copies est la négativité du **carré swap-face de la somme directe**. Prémices : Définitions 1.1-1.6, Théorèmes 18.1-18.4, Open 18.5-18.6, Thm 42.1, Thm 43.1. Utilisé par le Théorème 47.1.

Dans le langage de la littérature : la fonction de Wigner d'un état est une quasi-distribution réelle sur l'espace des phases, de marginales les distributions de position et d'impulsion ; pour un système fini de dimension d, une fonction sur les d×d cellules, qui **peut être négative** ; le recouvrement de deux états est l'intégrale du produit de leurs fonctions de Wigner **[82]**. L'échange de N copies identiques (bosons ou fermions) entre dans la distribution jointe par les termes d'interférence, au signe des statistiques **[76]**.


## 1. L'énoncé : deux objets, le même signe

Le théorème énonce cinq volets, tous **mesurés dans la source** :

1. Les deux objets — marginales de Wigner et carré swap-face — portent **le même signe** à chacun des cinq comptes de copies, de N = 1 à la borne parente 3.
2. Même **premier N négatif : 3**.
3. Le total de la distribution signée égale le **carré swap-face plus la lecture de clôture** aux quatre toiles jointes tabulées.
4. Une **seconde route**, depuis les liens à N corps, reproduit les grilles parentes **cellule par cellule**.
5. Le premier N auquel les deux objets *partent* : **jamais atteint** (`N part = none`).


## 2. La double paire : la plus petite toile jointe

La plus petite toile jointe de deux copies où se lit le carré swap-face est la **double paire** : deux copies du corps à deux docks du chapitre 42, appariées aux deux docks par deux liens d'appariement de type (2, 2), aucun lien frappé, habillage 9.


In [1]:
// Outils du carnet : BigInteger pour les comptes, Rationnel exact pour l'anneau Q(rac2, rac3, rac6).
// BCL seule : System.Numerics et System.Globalization -- aucun paquet externe, aucun flottant.
using System;
using System.Globalization;
using System.Numerics;

CultureInfo.DefaultThreadCurrentCulture = CultureInfo.InvariantCulture;

// Rationnel exact reduit : numerateur et denominateur en BigInteger, denominateur strictement positif.
sealed class Rationnel
{
    public BigInteger Numerateur { get; }
    public BigInteger Denominateur { get; }

    public Rationnel(BigInteger numerateur, BigInteger denominateur)
    {
        if (denominateur == BigInteger.Zero)
            throw new ArgumentException("Denominateur nul");
        if (denominateur < BigInteger.Zero)
        {
            numerateur = -numerateur;
            denominateur = -denominateur;
        }
        BigInteger pgcd = BigInteger.GreatestCommonDivisor(BigInteger.Abs(numerateur), denominateur);
        Numerateur = numerateur / pgcd;
        Denominateur = denominateur / pgcd;
    }

    public static Rationnel Entier(BigInteger n) => new Rationnel(n, BigInteger.One);

    public static Rationnel operator +(Rationnel x, Rationnel y) =>
        new Rationnel(
            x.Numerateur * y.Denominateur + y.Numerateur * x.Denominateur,
            x.Denominateur * y.Denominateur
        );

    public static Rationnel operator *(Rationnel x, Rationnel y) =>
        new Rationnel(x.Numerateur * y.Numerateur, x.Denominateur * y.Denominateur);

    public bool EstNul => Numerateur == BigInteger.Zero;

    public override string ToString() =>
        Denominateur == BigInteger.One
            ? Numerateur.ToString()
            : $"{Numerateur}/{Denominateur}";
}

// Controle executable : affiche CONFORME / ECART et leve une exception des que la valeur differe de l'attendu.
static class Controles
{
    public static void Verifier(string libelle, BigInteger obtenu, BigInteger attendu)
    {
        bool conforme = obtenu == attendu;
        Console.WriteLine(
            $"{libelle} = {obtenu} (attendu {attendu}) -> {(conforme ? "CONFORME" : "ECART")}"
        );
        if (!conforme)
            throw new Exception($"Ecart sur {libelle} : obtenu {obtenu}, attendu {attendu}");
    }
}

Console.WriteLine("Outils du carnet : BigInteger pour les comptes, Rationnel (BigInteger/BigInteger) pour l'anneau Q(rac2, rac3, rac6).");
Console.WriteLine("Aucune egalite flottante.");


Outils du carnet : BigInteger pour les comptes, Rationnel (BigInteger/BigInteger) pour l'anneau Q(rac2, rac3, rac6).


Aucune egalite flottante.


### Lecture du résultat
Aucune bibliothèque numérique : les comptes se font en `BigInteger` (précision arbitraire) et les coefficients de l'anneau en `Rationnel` exacte (numérateur et dénominateur `BigInteger`, fraction réduite) — chaque égalité affichée est une **vérification**, pas une approximation. Les éléments de l'anneau Q(√2, √3, √6) sont portés par leurs coefficients rationnels.


In [2]:
// Equation 68 : premieres puissances des trois blocs de la double paire.
// Regle de premiere puissance : habillage x somme des mesures des liens non frappes.
BigInteger habillage = 9;
BigInteger S1_c1 = habillage * (67 + 6 + 6);     // chaque copie, deux liens docks non frappes
BigInteger S1_c2 = habillage * (67 + 6 + 6);
BigInteger S1_appariement = habillage * (4 + 4); // les deux liens d'appariement ensemble
BigInteger S1 = S1_c1 + S1_c2 + S1_appariement;

Controles.Verifier("S1(c1) = 9 * (67+6+6)", S1_c1, 711);
Controles.Verifier("S1(c2)", S1_c2, 711);
Controles.Verifier("S1(appariement) = 9 * 8", S1_appariement, 72);
Controles.Verifier("S1 total", S1, 1494);


S1(c1) = 9 * (67+6+6) = 711 (attendu 711) -> CONFORME


S1(c2) = 711 (attendu 711) -> CONFORME


S1(appariement) = 9 * 8 = 72 (attendu 72) -> CONFORME


S1 total = 1494 (attendu 1494) -> CONFORME


### Lecture du résultat
La règle de première puissance (habillage × somme des mesures des liens non frappés) donne aux deux copies 9·79 = 711 chacune — 67 pour le corps, deux fois 6 pour les docks — et 9·8 = 72 aux deux liens d'appariement. Le total 1494 est la première puissance de la toile jointe entière : trois blocs, aucune frappe.


In [3]:
// Equation 69 : somme de paire et carre swap-face.
// Regle de paire : somme de paire = carre de la premiere puissance ;
// carre swap-face = carre lave (somme des carres de blocs)
//                 + termes croises avec les liens d'appariement
//                 - termes croises entre copies (chaque terme croise entre copies au signe -1).
BigInteger P2 = S1 * S1;
BigInteger carre_lave = S1_c1 * S1_c1 + S1_c2 * S1_c2 + S1_appariement * S1_appariement;
BigInteger croix_appariement = 2 * 2 * S1_appariement * S1_c1; // deux liens, deux copies
BigInteger croix_copies = 2 * S1_c1 * S1_c2;                     // signe -1
BigInteger Q_swap = carre_lave + croix_appariement - croix_copies;

Controles.Verifier("P2 = 1494^2", P2, 2232036);
Controles.Verifier("carre lave = 711^2+711^2+72^2", carre_lave, 1016226);
Controles.Verifier("croix appariement = 2*2*72*711", croix_appariement, 204768);
Controles.Verifier("croix copies (signe -1)", croix_copies, 1011042);
Controles.Verifier("Q_swap", Q_swap, 209952);
Console.WriteLine($"croix copies (signe -1)        = {-croix_copies}");


P2 = 1494^2 = 2232036 (attendu 2232036) -> CONFORME


carre lave = 711^2+711^2+72^2 = 1016226 (attendu 1016226) -> CONFORME


croix appariement = 2*2*72*711 = 204768 (attendu 204768) -> CONFORME


croix copies (signe -1) = 1011042 (attendu 1011042) -> CONFORME


Q_swap = 209952 (attendu 209952) -> CONFORME


croix copies (signe -1)        = -1011042


### Lecture du résultat
Le carré swap-face assemble trois morceaux : le **carré lavé** 1016226 (chaque bloc au carré, sommé), les **termes croisés d'appariement** +204768 (les deux liens de type (2, 2) croisés avec les deux copies), et les **termes croisés entre copies** soustraits — 1011042 — chaque terme croisé entre copies portant le signe −1, le signe de l'échange. La soustraction domine : 209952, bien en dessous du carré lavé. C'est cette structure — une somme de carrés moins les croix d'échange — que la source invoque pour expliquer le tour de signe à trois copies (§4).


## 3. La toile paire : le carrefour avec le chapitre 42

À la toile paire de deux copies — **un seul** lien d'appariement, les liens docks non appariés frappés, blocs 657, 657 et 36 — la même règle donne le carré swap-face de l'équation 71. Les briques de ce calcul sont les comptes de paires du chapitre 42, déjà rencontrés au carnet Chaînes-03.


In [4]:
// Equation 71 : carre swap-face a la toile paire (blocs 657, 657, 36).
BigInteger b1 = 657,
    b2 = 657,
    b3 = 36;

BigInteger carre_lave_pair = b1 * b1 + b2 * b2 + b3 * b3;
BigInteger croix_appariement_pair = 2 * 2 * b3 * b1;
BigInteger croix_copies_pair = 2 * b1 * b2;
BigInteger Q_swap_pair = carre_lave_pair + croix_appariement_pair - croix_copies_pair;

Controles.Verifier("carre lave = 657^2+657^2+36^2", carre_lave_pair, 864594);
Controles.Verifier("croix appariement = 2*2*36*657", croix_appariement_pair, 94608);
Controles.Verifier("croix copies (signe -1)", croix_copies_pair, 863298);
Controles.Verifier("Q_swap(paire)", Q_swap_pair, 95904);
Console.WriteLine($"croix copies (signe -1)        = {-croix_copies_pair}");

// Le carre swap-face de la toile paire n'est pas le terme croix-copies : la distinction est verifiee.
if (Q_swap_pair == croix_copies_pair)
    throw new Exception("Le carre swap-face de la toile paire ne doit pas etre confondu avec le terme croix-copies.");
Console.WriteLine("Distinction : Q_swap(paire) != croix copies (863298 est le terme crois-copies et le compte cite du ch. 42).");

// Carrefour avec le carnet Chaines-03 (ch. 42) : les memes entiers servent les deux objets.
var comptes_ch42 = new Dictionary<string, BigInteger>
{
    { "carre lave (hors 36^2)", 863298 },
    { "croix appariement", 94608 },
    { "carre de 36", 1296 },
};
Console.WriteLine("Carrefour ch. 42 / Chaines-03 : ");
foreach (var compte in comptes_ch42)
    Console.WriteLine($"  {compte.Key} = {compte.Value}");
Controles.Verifier("carre lave (hors 36^2) = 657^2+657^2", b1 * b1 + b2 * b2, comptes_ch42["carre lave (hors 36^2)"]);
Controles.Verifier("carre de 36 = 36^2", b3 * b3, comptes_ch42["carre de 36"]);


carre lave = 657^2+657^2+36^2 = 864594 (attendu 864594) -> CONFORME


croix appariement = 2*2*36*657 = 94608 (attendu 94608) -> CONFORME


croix copies (signe -1) = 863298 (attendu 863298) -> CONFORME


Q_swap(paire) = 95904 (attendu 95904) -> CONFORME


croix copies (signe -1)        = -863298


Distinction : Q_swap(paire) != croix copies (863298 est le terme crois-copies et le compte cite du ch. 42).


... (2 outputs left out)

  croix appariement = 94608


  carre de 36 = 1296


carre lave (hors 36^2) = 657^2+657^2 = 863298 (attendu 863298) -> CONFORME


carre de 36 = 36^2 = 1296 (attendu 1296) -> CONFORME


### Lecture du résultat
Le calcul de la toile paire réutilise **les mêmes entiers que le chapitre 42** — 863298, 94608, 1296, les comptes de paires déjà rejoués au carnet Chaînes-03. C'est le carrefour de ce chapitre : la toile jointe de deux copies n'est pas un objet neuf, elle est la toile de mesure du chapitre 42 lue par la règle de paire. Le résultat 95904 est positif : à deux copies, aucun signe négatif.


### Exercice 1 — Prévoir le signe d'une toile voisine

On donne la règle : `Q_swap = (somme des carrés de blocs) + (croix d'appariement) − (croix entre copies, signe −1)`.

1. Avec les blocs 711, 711 et 72 (la double paire), recalculer `Q_swap` pas à pas et retrouver 209952.
2. Remplacer le bloc d'appariement 72 par 36 (un seul lien) et observer le sens de variation de `Q_swap`.
3. Le remplacement ne change ni le nombre de blocs ni le nombre de copies : dire si le signe de `Q_swap` change, et ce que ce calcul à deux copies permet — ou ne permet **pas** — de conclure sur le signe des toiles de trois copies.


In [5]:
// Exercice 1 : le signe d'une toile voisine.
// 1. Recalculer Q_swap avec les blocs 711, 711, 72.
// 2. Remplacer 72 par 36 et observer le sens de variation.
// 3. Dire si le signe change, et ce que ce calcul a deux copies permet ou ne permet pas de conclure a trois copies.

// TODO etudiant : implémenter la règle de paire ci-dessous, puis l'appeler sur les deux toiles.
// TODO etudiant : le nombre de liens d'appariement est un paramètre de la fonction.
BigInteger QSwap(BigInteger blocCopie1, BigInteger blocCopie2, BigInteger blocAppariement, int nbLiensAppariement)
{
    // TODO etudiant : somme des carrés de blocs + croix d'appariement - croix entre copies (signe -1).
    return BigInteger.Zero;
}

// TODO etudiant : appeler QSwap sur (711, 711, 72) puis sur (711, 711, 36), comparer les valeurs
// TODO etudiant : et conclure sur le signe (point 3 de l'énoncé).
Console.WriteLine("Exercice 1 a completer");


Exercice 1 a completer


## 4. À trois copies : le signe tourne

Aux trois copies — la **chaîne** (blocs 2133, 1971 et 1971, habillage 27, deux liens d'appariement 216 ensemble) et le **triangle** (blocs 2133, 2133 et 2133, trois liens d'appariement 324 ensemble) — la source donne des carrés swap-face **négatifs** (équation 72). Ces deux valeurs sont **citées** : le carnet ne les re-dérive pas ici — la partie exacte travaillée sur cette page est celle à deux copies, et la règle à trois corps n'est pas réécrite. Le premier N négatif est le même aux deux objets :

`N_neg(Wigner) = 3 = N_neg(swap)`, `N part = none`.


In [6]:
// Equation 72 : le premier N negatif, le meme aux deux objets.
// Donnees SAISIES : les carres swap-face a N=3 sont CITES (ch. 46) -- ils ne sont pas re-derives ici.
// N=2 : la double paire, valeur rejouee plus haut (eq. 68-69).
var Q_swap_par_N = new Dictionary<int, BigInteger>
{
    { 2, Q_swap },                                  // double paire : rejouee (eq. 69)
    { 3, BigInteger.Parse("-9595827") },          // chaine (CITE)
};
BigInteger Q_swap_triangle3 = BigInteger.Parse("-9397539"); // triangle (CITE)

// Comparaison exacte des valeurs citees.
Controles.Verifier("N=2 : Q_swap(double paire) [rejoue]", Q_swap_par_N[2], 209952);
Controles.Verifier("N=3 : Q_swap(chaine)  [CITE]", Q_swap_par_N[3], BigInteger.Parse("-9595827"));
Controles.Verifier("N=3 : Q_swap(triangle) [CITE]", Q_swap_triangle3, BigInteger.Parse("-9397539"));

// Signes calcules sur les valeurs ci-dessus.
bool positif_N2 = Q_swap_par_N[2] > BigInteger.Zero;
bool negatif_chaine = Q_swap_par_N[3] < BigInteger.Zero;
bool negatif_triangle = Q_swap_triangle3 < BigInteger.Zero;
Console.WriteLine($"N=2 : Q_swap(double paire) = {Q_swap_par_N[2]} -> positif ? {positif_N2}");
Console.WriteLine($"N=3 : Q_swap(chaine)       = {Q_swap_par_N[3]} -> negatif ? {negatif_chaine}");
Console.WriteLine($"N=3 : Q_swap(triangle)     = {Q_swap_triangle3} -> negatif ? {negatif_triangle}");

// Grandeur DERIVEE : le premier N negatif du carre swap-face, lu sur les valeurs saisies.
int premier_negatif_swap = -1;
foreach (var entree in Q_swap_par_N)
    if (
        entree.Value < BigInteger.Zero
        && (premier_negatif_swap < 0 || entree.Key < premier_negatif_swap)
    )
        premier_negatif_swap = entree.Key;
Controles.Verifier("Premier N negatif, swap-face (derive)", premier_negatif_swap, 3);

// Donnee SAISIE : les marginales de Wigner sont negatives a N=3, pas a N=2 (CITE).
int premier_negatif_wigner = 3;
Console.WriteLine($"Premier N negatif, swap-face  : {premier_negatif_swap} (derive)");
Console.WriteLine($"Premier N negatif, Wigner     : {premier_negatif_wigner} (CITE)");
Controles.Verifier("N_neg(Wigner) = N_neg(swap)", premier_negatif_wigner, premier_negatif_swap);
Console.WriteLine("Premier N de parting          : aucun (N part = none, eq. 72)");


N=2 : Q_swap(double paire) [rejoue] = 209952 (attendu 209952) -> CONFORME


N=3 : Q_swap(chaine)  [CITE] = -9595827 (attendu -9595827) -> CONFORME


N=3 : Q_swap(triangle) [CITE] = -9397539 (attendu -9397539) -> CONFORME


N=2 : Q_swap(double paire) = 209952 -> positif ? True


N=3 : Q_swap(chaine)       = -9595827 -> negatif ? True


N=3 : Q_swap(triangle)     = -9397539 -> negatif ? True


Premier N negatif, swap-face (derive) = 3 (attendu 3) -> CONFORME


Premier N negatif, swap-face  : 3 (derive)


Premier N negatif, Wigner     : 3 (CITE)


N_neg(Wigner) = N_neg(swap) = 3 (attendu 3) -> CONFORME


Premier N de parting          : aucun (N part = none, eq. 72)


### Lecture du résultat
Entre deux et trois copies, le signe tourne — aux **deux** toiles de trois copies à la fois (chaîne −9595827, triangle −9397539). C'est le cœur du théorème : la négativité des marginales de Wigner au-delà de deux copies **est** la négativité du carré swap-face — même premier N négatif (3), aucun parting sur toute la plage mesurée. La source explique ce basculement par la structure : chaque copie ajoutée multiplie les termes croisés entre copies, tous au signe −1, jusqu'à dominer les carrés lavés — explication **citée** : le carnet, qui mesure jusqu'à deux copies, ne la dérive pas.


In [7]:
// Equation 70 : la lecture de cloture, element exact de l'anneau Q(rac2, rac3, rac6).
// Portee par ses coefficients rationnels exacts (la partie a deux echanges,
// 254 enregistrements de cloture) :
var R_cl = new Dictionary<string, Rationnel>
{
    { "1", Rationnel.Entier(BigInteger.Parse("259966108392949120000")) },
    { "rac2", Rationnel.Entier(1286225) },
    { "rac3", Rationnel.Entier(5977541147456000) },
    { "rac6", Rationnel.Entier(0) },
};
int enregistrements_cloture = 254;

foreach (var (nomBase, coef) in R_cl)
{
    string signe = coef.Numerateur >= BigInteger.Zero ? "+" : "-";
    Console.WriteLine($"coefficient de {nomBase,-5} : {signe}{BigInteger.Abs(coef.Numerateur)}");
}

// Denominateurs : distincts, tries, sans LINQ.
var denominateurs = new List<BigInteger>();
foreach (var coef in R_cl.Values)
    if (!denominateurs.Contains(coef.Denominateur))
        denominateurs.Add(coef.Denominateur);
denominateurs.Sort();
Console.WriteLine(
    $"Denominateurs (tous 1 : coefficients entiers de l'anneau) : {string.Join(", ", denominateurs)}"
);

// Controles executables : coefficient de rac6 nul, quatre coefficients, denominateurs egaux a 1.
Controles.Verifier("coefficient de rac6 (nul)", R_cl["rac6"].Numerateur, BigInteger.Zero);
Controles.Verifier("nombre de coefficients de l'anneau", R_cl.Count, 4);
foreach (var coef in R_cl.Values)
    if (coef.Denominateur != BigInteger.One)
        throw new Exception("Un coefficient de l'anneau doit porter un denominateur egal a 1.");
Console.WriteLine("Tous les denominateurs valent 1 : CONFORME");

Console.WriteLine($"Enregistrements de cloture sommes : {enregistrements_cloture}");
Console.WriteLine("Marginales de cloture non negatives aux 22 sites (CITE) ; total = Q_swap + R_cl (eq. 71, CITE)");
Console.WriteLine("R_cl est une cellule des tables de somme directe, re-computee sur la page not (CITE)");


coefficient de 1     : +259966108392949120000


coefficient de rac2  : +1286225


coefficient de rac3  : +5977541147456000


coefficient de rac6  : +0


Denominateurs (tous 1 : coefficients entiers de l'anneau) : 1


coefficient de rac6 (nul) = 0 (attendu 0) -> CONFORME


nombre de coefficients de l'anneau = 4 (attendu 4) -> CONFORME


Tous les denominateurs valent 1 : CONFORME


Enregistrements de cloture sommes : 254


Marginales de cloture non negatives aux 22 sites (CITE) ; total = Q_swap + R_cl (eq. 71, CITE)


R_cl est une cellule des tables de somme directe, re-computee sur la page not (CITE)


### Lecture du résultat
La lecture de clôture vit dans l'anneau Q(√2, √3, √6) : quatre coefficients rationnels, dont un nul (√6). Elle est la seconde partie de la somme directe — le total de la distribution signée est toujours **Q_swap + R_cl** (équation 71), et ses marginales de clôture restent non négatives aux vingt-deux sites. Le signe négatif du chapitre ne vient donc jamais de la clôture : il vient du carré swap-face, c'est-à-dire de l'échange.


### Exercice 2 — Lire un élément de l'anneau

On donne `R_cl` par ses quatre coefficients (`Rationnel`), dans le dictionnaire construit à la cellule précédente.

1. Écrire une fonction `Echelle(elem, k)` qui multiplie chaque coefficient par l'entier `k`.
2. Vérifier que `Echelle(R_cl, 2)` a bien tous ses coefficients pairs.
3. Un élément de l'anneau est **nul** si et seulement si ses quatre coefficients sont nuls — expliquer pourquoi la comparaison coefficient par coefficient suffit, sans jamais évaluer les racines.


In [8]:
// Exercice 2 : manipuler un element de l'anneau Q(rac2, rac3, rac6).
// 1. Ecrire Echelle(elem, k) : multiplication de chaque coefficient par k.
// 2. Verifier que Echelle(R_cl, 2) a tous ses coefficients pairs.
// 3. Expliquer pourquoi la nullite se lit coefficient par coefficient.

// TODO etudiant : multiplier chaque coefficient de elem par k, en conservant les quatre bases.
// TODO etudiant : verifier que les coefficients de Echelle(R_cl, 2) sont tous pairs.
// TODO etudiant : conclure sur la nullite coefficient par coefficient (point 3 de l'enonce).
Dictionary<string, Rationnel> Echelle(Dictionary<string, Rationnel> elem, BigInteger k)
{
    // TODO etudiant : corps a completer -- la valeur rendue ici est neutre.
    return new Dictionary<string, Rationnel>();
}

Console.WriteLine("Exercice 2 a completer");


Exercice 2 a completer


## 5. Le census et le résultat ouvert : quatre constats sur cinq

Le census (cité) : énumération exhaustive des cellules de la distribution signée aux quatre toiles jointes tabulées — le décompte des cellules à la double paire est imprimé par la cellule de census qui suit, chacune élément exact de l'anneau ; les **deux routes** d'accord cellule par cellule ; totaux de paires égaux aux sommes de paires aux **six comparaisons** ; contre-test : frapper le signe déplace le premier N négatif à la route principale (au signe +, aucun compte négatif) ; **douze contrôles** aux deux moteurs ; second moteur ; Lean énumère « 6 objects ».

Le résultat ouvert tient la fonction de Wigner comme cellule swap-face à **quatre des cinq constats** — et pas à la cinquième : le **recouvrement** de deux distributions part du recouvrement d'enregistrements à la somme directe, aux deux paires testées (l'électron avec lui-même, l'électron avec le corps muon). L'entrée de la fonction de Wigner type donc une **glose à sa cinquième étape**.


In [9]:
// Les cinq constats du census (CITES, ch. 46) : saisis, non derives ici -- 4 tiennent, le 5e fait la glose.
var constats_census = new Dictionary<string, bool>
{
    {
        "corps electron (ch. 42) : 3685 enregistrements, toute cellule non negative, marginales = lecture lavee",
        true
    },
    {
        "decomposition en cellules aux 4 toiles, croix entre copies au signe -1, paire = Q_swap, cloture = R_cl",
        true
    },
    {
        "marginales negatives a N=3, pas a N=2 ; negativite hors des seules cellules jointes",
        true
    },
    {
        "recouvrement de deux distributions = recouvrement d'enregistrements (ch. 46 : part aux deux paires testees, electron avec lui-meme et electron avec le corps muon)",
        false
    },
    { "total = Q_swap + R_cl aux quatre toiles", true },
};
foreach (var constat in constats_census)
    Console.WriteLine($"{(constat.Value ? "TIENT  " : "PART   ")} {constat.Key}");

int nb_tiennent = 0;
bool echec_sur_recouvrement = false;
foreach (var constat in constats_census)
{
    if (constat.Value)
        nb_tiennent++;
    else
        echec_sur_recouvrement = constat.Key.Contains("recouvrement");
}
Console.WriteLine(
    $"Constats qui tiennent : {nb_tiennent} sur {constats_census.Count} -> la cellule Wigner est une GLOSSE a sa 5e etape"
);

// Controle calculé sur la saisie des constats : quatre tiennent, le cinquieme echoue sur le recouvrement.
Controles.Verifier("nombre de constats saisis", constats_census.Count, 5);
Controles.Verifier("constats qui tiennent", nb_tiennent, 4);
if (!echec_sur_recouvrement)
    throw new Exception("Le cinquieme constat (recouvrement) doit etre celui qui part.");
Console.WriteLine("Echec porte sur le recouvrement : CONFORME");

// Chiffres du census (CITES, ch. 46) :
Console.WriteLine("Census : 18 cellules a la double paire ; 6 comparaisons de totaux ; 12 controles aux deux moteurs ;");
Console.WriteLine("         contre-test : frapper le signe deplace le premier N negatif (une cellule) ; Lean : 6 objects enumerated");
Console.WriteLine("Portee : donnees citees du ch. 46 ; ce controle porte sur la saisie, il n'est pas une preuve exhaustive nouvelle.");


TIENT   corps electron (ch. 42) : 3685 enregistrements, toute cellule non negative, marginales = lecture lavee


TIENT   decomposition en cellules aux 4 toiles, croix entre copies au signe -1, paire = Q_swap, cloture = R_cl


TIENT   marginales negatives a N=3, pas a N=2 ; negativite hors des seules cellules jointes


PART    recouvrement de deux distributions = recouvrement d'enregistrements (ch. 46 : part aux deux paires testees, electron avec lui-meme et electron avec le corps muon)


TIENT   total = Q_swap + R_cl aux quatre toiles


Constats qui tiennent : 4 sur 5 -> la cellule Wigner est une GLOSSE a sa 5e etape


... (2 outputs left out)

Echec porte sur le recouvrement : CONFORME


Census : 18 cellules a la double paire ; 6 comparaisons de totaux ; 12 controles aux deux moteurs ;


         contre-test : frapper le signe deplace le premier N negatif (une cellule) ; Lean : 6 objects enumerated


Portee : donnees citees du ch. 46 ; ce controle porte sur la saisie, il n'est pas une preuve exhaustive nouvelle.


### Lecture du résultat
Quatre constats du census sur cinq font de la fonction de Wigner la cellule de lecture swap de la somme directe — et le cinquième échoue sur le **recouvrement** : l'intégrale du produit de deux fonctions de Wigner [82] ne reproduit le recouvrement d'enregistrements qu'avant la somme directe (aux deux paires testées : électron–électron, électron–muon). C'est le statut exact du théorème : **conditionnel** sur la fonction de Wigner comme cellule swap — une correspondance qui tient partout sauf au recouvrement, et qui y devient glose. Le théorème lui-même (le signe à N corps) est démontré sous cette hypothèse.


### Exercice 3 — Faire tourner un contre-test

Le contre-test du census : frapper le signe (remplacer le −1 des croix entre copies par +1) déplace le premier N négatif — au signe +, **aucun** compte de copies n'est négatif.

1. Recalculer le `Q_swap` de la double paire (blocs 711, 711, 72) avec le signe +1 sur les croix entre copies.
2. Comparer au `Q_swap` d'origine (209952) et expliquer l'écart.
3. Ce calcul à deux copies : dire ce qu'il vérifie exactement, et s'il suffit — ou non — à conclure quoi que ce soit au signe à trois copies. Relier ce calcul au constat n°3 de `constats_census` (« marginales negatives a N=3, pas a N=2 ») : c'est ce constat, et le contre-test à N=3 qu'il résume, qui portent la négativité à trois copies — tous deux **cités de la source**, le carnet, qui mesure jusqu'à deux copies, ne les dérive pas.


In [10]:
// Exercice 3 : le contre-test du signe.
// 1. Recalculer Q_swap (711, 711, 72) avec le signe +1 sur les croix entre copies.
// 2. Comparer a 209952 et expliquer l'ecart.
// 3. Ce calcul a deux copies : dire ce qu'il verifie, et ce qu'il ne permet PAS de conclure
//    au signe a trois copies. Le constat deplace (N=3) et le contre-test sont CITES de la source.

// TODO etudiant : signer les termes croises entre copies par un parametre (le signe de l'echange).
// TODO etudiant : appeler la fonction sur les blocs 711, 711, 72, puis comparer au Q_swap d'origine.
// TODO etudiant : conclure sur la portee du calcul a deux copies (point 3 de l'enonce).
BigInteger QSwapContreTest(BigInteger blocCopie1, BigInteger blocCopie2, BigInteger blocAppariement, int signeCroixCopies)
{
    // TODO etudiant : corps a completer -- la valeur rendue ici est neutre.
    return BigInteger.Zero;
}

Console.WriteLine("Exercice 3 a completer");


Exercice 3 a completer


## Conclusion

Ce que ce carnet a rejoué : les premières puissances de la double paire (711, 72, total 1494), la somme de paire 2232036, les carrés swap-face **209952** (double paire) et **95904** (toile paire — les entiers du chapitre 42, carrefour avec Chaînes-03), la lecture de clôture comme élément exact de Q(√2, √3, √6), et le basculement du signe à trois copies (chaîne −9595827, triangle −9397539, cités).

Ce que le théorème établit, sous sa condition : la négativité des marginales de Wigner au-delà de deux copies **est** la négativité du carré swap-face de la somme directe — même signe aux cinq comptes de copies, même premier N négatif (3), total = Q_swap + R_cl aux quatre toiles, seconde route d'accord cellule par cellule, aucun parting. Ce qui reste ouvert : la fonction de Wigner comme cellule swap tient à quatre constats du census sur cinq — au recouvrement, elle *part*, et l'entrée type une glose. Le signe à N corps, lui, est démontré ; c'est son pont vers la fonction de Wigner de la littérature [82] qui garde une marche manquante.


## Références

- **Source** : compendium (gisement privé, copie archivée — jamais commise), chapitre 46, Théorème 46.1 « The Wigner function and the many-body sign », pages 126-128. Équations 68-72.
- **Correspondances citées par nom** : [82] fonction de Wigner — quasi-distribution réelle sur l'espace des phases, marginales = position et impulsion, recouvrement de deux états = intégrale du produit de leurs fonctions de Wigner ; [76] l'échange de N copies identiques entre dans la distribution jointe par les termes d'interférence, au signe des statistiques.
- **Chapitres en correspondance par nom** : 35, 43, 45, 57, « The Dirac equation at the dock » (ch. I.31), 159, 162, 176, 185, 238, 239 ; l'électron et le muon comme quanta de spin un demi d'un champ, masse et spin comme étiquettes de Lorentz.
- **Prémices** : Déf 1.1-1.6, Thm 18.1-18.4, Open 18.5-18.6, Thm 42.1 (Chaînes-03), Thm 43.1 (Chaînes-04). Utilisé par Thm 47.1. La plus longue chaîne de prémices du compendium fait douze chapitres.
- **Carnets de la série** : Chaînes-01 (théorie des chaînes), Chaînes-02 (strate 18), Chaînes-03 (ch. 42, mesure), Chaînes-04 (ch. 43, indiscernabilité), Chaînes-05 (ch. 44, énergie), Chaînes-06 (ch. 45, intégrale de chemin). Lean : « 6 objects enumerated » au census de ce chapitre.


## À retenir

1. **Le signe à N corps n'est pas un objet nouveau** : la négativité des marginales de Wigner au-delà de deux copies **est** la négativité du carré swap-face de la somme directe (Thm 46.1, conditionnel à la source). Même signe aux cinq comptes de copies, même premier $N$ négatif (3), aucun parting.
2. **L'arithmétique de la double paire est rejouée, pas recopiée** : $S_1 = 9 \cdot (67 + 6 + 6) = 711$ par copie, $9 \cdot 8 = 72$ pour l'appariement, total **1494** ; $P_2 = 1494^2 = 2232036$ ; $Q_{\text{swap}} = 1016226 + 204768 - 1011042 = 209952$.
3. **Le carrefour avec Chaînes-03 est réel, et c'est le même calcul** : à la toile paire, $864594 + 94608 - 863298 = 95904$ — les entiers $863298$, $94608$, $1296$ du chapitre 42 resservent tels quels. Les deux carnets mesurent la même structure à deux échelles.
4. **La lecture de clôture (éq. 70) est un élément exact de $\mathbb{Q}(\sqrt2,\sqrt3,\sqrt6)$** : le carnet l'obtient en `Rationnel` (numérateur et dénominateur `BigInteger`), sans jamais passer par un flottant.
5. **Le signe à trois copies est cité, pas re-dérivé** : chaîne $-9595827$, triangle $-9397539$ sont étiquetés `(CITES)` — l'extrapolation à trois corps est hors du périmètre mesuré, et le carnet le dit plutôt que de le maquiller.
6. **Ce qui reste ouvert** : la fonction de Wigner comme cellule swap tient **quatre constats du census sur cinq** — au recouvrement de deux distributions, elle *part*, et la cinquième entrée n'est qu'une glose. Le pont vers la fonction de Wigner de la littérature [82] garde une marche manquante.


## Vérifiez votre compréhension

1. Comment se décompose le total $1494$, et laquelle de ses trois composantes la source donne-t-elle déjà ?
2. Que vaut $Q_{\text{swap}}$ de la double paire, et par quelle soustraction le carnet l'obtient-il ?
3. Pourquoi le carrefour avec Chaînes-03 n'est-il pas une coïncidence de vocabulaire ?
4. Sur combien de constats du census la fonction de Wigner comme cellule swap tient-elle, et où *part*-elle ?

<details>
<summary>Réponses</summary>

1. $S_1 = 9 \cdot (67 + 6 + 6) = 711$ **par copie**, plus $9 \cdot 8 = 72$ pour l'appariement, soit $711 + 711 + 72 = 1494$. Les trois entiers $67$, $6$, $6$ sont donnés par la source ; c'est leur combinaison qui est rejouée ici.
2. $Q_{\text{swap}} = 1016226 + 204768 - 1011042 = 209952$ : le carré lavé, plus la croix d'appariement, moins la croix entre copies — la règle de paire écrite en clair au §2.
3. Parce que la toile paire redonne littéralement les entiers du chapitre 42 : $864594 + 94608 - 863298 = 95904$, où $863298$, $94608$ et $1296$ sont ceux que Chaînes-03 ferme par $1350^2$. Le même calcul, à une autre échelle, n'est pas une analogie.
4. Sur **quatre sur cinq**. Elle tient aux cellules non négatives, à la décomposition en cellules, aux marginales négatives à $N = 3$ et à l'identité total $=$ $Q_{\text{swap}} + R_{cl}$ ; elle *part* au recouvrement de deux distributions, qui n'est pas un recouvrement d'enregistrements. La cinquième entrée, elle, est une glose et non un constat.

</details>


## Pour aller plus loin

*Chaînes-08 — les corollaires quantiques à leurs racines établies*
reprend ce carnet là où il s'arrête : le théorème de Wigner y devient le **premier nommage** du
chapitre 47, et sa clause manquante y est explicitement la face swap comme cellule. C'est la suite
directe du pont laissé ouvert ici.

En amont, [Chaînes-03 — la mesure](Chaines-03-Mesure-Lavage-POVM-Python.ipynb) fournit les entiers
du chapitre 42 que ce carnet réutilise au carrefour de la toile paire, et
[Chaînes-04 — l'indiscernabilité et le signe spin-statistique](Chaines-04-Indiscernabilite-Spin-Statistique-Python.ipynb)
donne les deux faces du signe d'échange dont le signe à N corps hérite.

Pour qui veut vérifier plutôt que lire : le census du §5 — 309 lectures, six types de paires,
32851 coups, treize corps, deux moteurs — est le point de reprise. C'est lui, et non le carnet,
qui porte les cinq constats ; quatre tiennent, le cinquième désigne exactement la marche
manquante.

**Version Python** de ce carnet : [Chaînes-07 — la fonction de Wigner et le signe à N corps (Python)](Chaines-07-Fonction-Wigner-Signe-N-Corps-Python.ipynb) — même fil, mêmes entrées, arithmétique reprise en `Fraction` exacte.

> Carnet issu de l'EPIC #26534.
